# 04 · Auditoría de labels del router (§8.6, REQ-13)

Hallazgo (2026-10-03, datos crudos):
- `call_transcripts.customer_text` son **42 plantillas** de consulta de saldo, repartidas igual entre las 6 categorías.
- `detected_intents` = `consulta_general` (o nulo) en el 100%.
- `reason_category` = `contact_reason` (6 valores) y la escalación es ~10% en todas: no hay señal.

→ El dataset no da labels de intención válidos. El corpus de entrenamiento es team_generated
(`ml/data/intent_corpus.jsonl`, CON-02) y esta auditoría es la evidencia de por qué.

In [ ]:
from common import ROOT, connect, save_result, show

con = connect()

In [ ]:
templates = show(con, "Diversidad de textos (dígitos enmascarados)", """
    SELECT count(*) AS filas, count(DISTINCT regexp_replace(customer_text, '[0-9]+', '#', 'g')) AS plantillas
    FROM call_transcripts""")

In [ ]:
intents = show(con, "detected_intents", """
    SELECT detected_intents, count(*) AS n FROM call_transcripts GROUP BY 1 ORDER BY n DESC""")

In [ ]:
by_cat = show(con, "Las mismas plantillas en todas las categorías", """
    SELECT i.reason_category, count(DISTINCT regexp_replace(t.customer_text, '[0-9]+', '#', 'g')) AS plantillas,
           count(*) AS n
    FROM call_transcripts t JOIN call_center_interactions i USING (interaction_id) GROUP BY 1 ORDER BY n DESC""")

## Corpus del equipo y muestra manual
Con `ml/data/intent_corpus.jsonl`: distribución por clase/idioma y acuerdo con `ml/labels/manual_labels.csv`.

In [ ]:
# analysis/ no depende de sofia-ml: se leen los archivos directo (mismo formato que sofia_ml.labeling).
corpus_path = ROOT / "ml" / "data" / "intent_corpus.jsonl"
if corpus_path.exists():
    show(con, "Corpus por clase e idioma", f"""
        SELECT label, language, origin, count(*) AS n, count(DISTINCT group_id) AS familias
        FROM read_json_auto('{corpus_path.as_posix()}') GROUP BY ALL ORDER BY label, language""")  # noqa: S608
else:
    print("todavía no hay corpus:", corpus_path)

In [ ]:
save_result("label_audit", {
    "transcript_templates": templates.to_dicts(),
    "detected_intents": intents.to_dicts(),
    "by_category": by_cat.to_dicts(),
})